# Mapless ESDF queries, end to end

This buffer answers signed distance queries straight from a ring of posed depth
images. There is no map: no voxel grid, no TSDF, no incremental fusion. A query
searches the images themselves, so a new frame is usable the moment it is
inserted and an old one is forgotten the moment it is overwritten.

Two distances come back for every point, and the difference between them is the
whole story of what a depth camera does and does not know:

| | space no camera has seen counts as | use it when |
|---|---|---|
| **optimistic** | free | you are willing to fly into the unknown |
| **conservative** | occupied | you are not |

The optimistic distance is the distance to the nearest point back projected from
a depth pixel. The conservative one also measures to the *boundary of unknown
space*: the surfaces behind occluding objects, the frustum walls, the near plane.
It is signed, so it goes negative once you are inside the unknown.

Everything below is generated. There is no dataset to download.

In [ ]:
import cuda_depth_buffer as cdb
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colors

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## 1. A scene and a camera

`AnalyticScene` is a handful of primitives a camera ray can be intersected with
in closed form, so a depth image is *computed*, not loaded. A forest is the case
this library is built for: thin trunks, most of the image far away, and plenty
of occlusion behind every trunk.

In [ ]:
intrinsics = cdb.CameraIntrinsics(
    fx=320.0, fy=320.0, cx=319.5, cy=239.5, width=640, height=480,
    z_near=0.3, z_far=15.0,
)

scene = cdb.AnalyticScene.forest(seed=7, num_cylinders=32, num_spheres=18, extent=12.0)

# A transect: the drone flies along x with the forest off to one side, looking
# sideways into it. That is what a mapping pass looks like, and unlike an orbit
# it leaves the far side of every trunk unseen. `cdb.orbit_poses` builds a ring
# instead, if that is the case you care about.
FLY_Y, FLY_Z = -9.0, 2.0
WINDOW = 16  # frames the buffer holds

flight = [cdb.look_at([x, FLY_Y, FLY_Z], [x, 0.0, 1.0])
          for x in np.linspace(-9.0, 9.0, 40)]

# Rendered once, on the CPU, so the slider further down can move the drone
# without waiting for ray casts. Takes a few seconds.
flight_depths = [scene.render(intrinsics, pose) for pose in flight]

# The figures below take the drone to the middle of its flight, which is also
# where the slider further down starts.
START = (len(flight) - WINDOW) // 2
poses = flight[START:START + WINDOW]
depths = flight_depths[START:START + WINDOW]
print(f"{len(flight)} poses along the flight, {WINDOW} of them in the buffer, "
      f"{intrinsics.width}x{intrinsics.height} each")

### The ribbon of images the buffer holds

This is the entire state of the system. Nothing is fused, nothing is
accumulated: these images and their poses *are* the map.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 3.6), layout="constrained")
for ax, depth, pose in zip(axes.ravel(), depths, poses):
    ax.imshow(depth, cmap="viridis_r", vmin=0.3, vmax=14.0, origin="upper")
    ax.set_title(f"({pose[0]:+.1f}, {pose[1]:+.1f})", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("the ring buffer, one tile per frame, near is bright")

## 2. Filling the buffer

`BufferParams` is the whole configuration, and the same struct drives the C++
examples and the ROS node. `num_frames` fixes the ring size for the life of the
buffer.

In [ ]:
params = cdb.BufferParams()
params.num_frames = len(poses)
params.pose_covariance_confidence = 0.0   # no pose uncertainty in this demo

buffer = cdb.CudaDepthBuffer(intrinsics, params)
buffer.warmup()

for i, (depth, pose) in enumerate(zip(depths, poses)):
    buffer.set_posed_depth_image(depth, np.asarray(pose, np.float32),
                                 stamp_ns=i * 62_500_000, is_keyframe=False)
print(buffer.params, buffer.intrinsics)

## 3. The merged field

Queries are points in the reference frame the poses are in, so a dense slice is
just a grid of them. The merge across all sixteen views happens inside the
query: each frame answers, and the answers are pooled.

In [ ]:
def slice_grid(extent=10.0, resolution=220, height=1.0):
    axis = np.linspace(-extent, extent, resolution, dtype=np.float32)
    xx, yy = np.meshgrid(axis, axis)
    points = np.column_stack([xx.ravel(), yy.ravel(),
                              np.full(xx.size, height, np.float32)]).astype(np.float32)
    return points, xx, yy, axis


def query_slice(buffer, truncation=3.0, **kwargs):
    points, xx, yy, axis = slice_grid(**kwargs)
    result = buffer.query(points, conservative=True, truncation_radius=truncation)
    return (result.optimistic_distance.reshape(xx.shape),
            result.conservative_distance.reshape(xx.shape), axis)


optimistic, conservative, axis = query_slice(buffer, truncation=3.0, height=1.0)
print(f"conservative: {np.nanmin(conservative):+.2f} to {np.nanmax(conservative):+.2f} m, "
      f"{100 * np.mean(conservative < 0):.0f}% of the slice is inside unknown space")

In [ ]:
# Red is free space, blue is unknown, white is the boundary between them. The
# optimistic channel is never negative, so it uses the free half of the same
# scale and comes out white to red.
FREE_UNKNOWN = plt.get_cmap("seismic")
FREE_ONLY = colors.LinearSegmentedColormap.from_list(
    "seismic_free", FREE_UNKNOWN(np.linspace(0.5, 1.0, 256)))


def draw_field(ax, field, axis, title, signed, cameras=None, limit=None):
    # The scale is symmetric about zero and the same on both sides, so a colour
    # means a magnitude and white always means the boundary. Stretching each
    # side to its own extreme would paint a metre of clearance the same red as
    # ten, and on a slice near the floor nothing is ever further than the floor.
    if limit is None:
        limit = float(np.nanmax(np.abs(field)))
    limit = max(limit, 1e-3)
    extent = [axis[0], axis[-1], axis[0], axis[-1]]
    if signed:
        image = ax.imshow(field, extent=extent, cmap=FREE_UNKNOWN, origin="lower",
                          vmin=-limit, vmax=limit)
    else:
        # Never negative, so it takes the free half of the same scale. A value
        # lands on the same colour either way.
        image = ax.imshow(field, extent=extent, cmap=FREE_ONLY, origin="lower",
                          vmin=0.0, vmax=limit)
    if cameras is None:
        cameras = poses
    ax.plot([p[0] for p in cameras], [p[1] for p in cameras], "o-", ms=3, lw=1,
            color="limegreen", label="camera")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.legend(loc="upper right", fontsize=7)
    plt.colorbar(image, ax=ax, shrink=0.82, label="distance [m]")


# One magnitude for both panels, so a colour means the same thing in each.
limit = max(np.nanmax(np.abs(optimistic)), np.nanmax(np.abs(conservative)))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
draw_field(axes[0], optimistic, axis, "optimistic: unknown space is free", False,
           limit=limit)
draw_field(axes[1], conservative, axis,
           "conservative: unknown space is occupied", True,
           limit=limit)
fig.tight_layout()

Read the right hand panel as a visibility map. Red is space some camera saw and
found empty. Blue is space no camera could see: inside the trunks, in the wedge
behind each of them, behind the camera, and past the far plane. White is where
the two meet, and it is exactly what a conservative planner must not cross.

The shadows all point away from the flight line, because that is the only place
anything looked from. Fly the other side too and they close.

The left panel has none of that structure, because the optimistic channel only
ever measures to something a camera actually hit. Its ceiling here is the
ground: on a slice one metre up, nothing is ever further than a metre from the
floor, which is why so much of it saturates. The dark band across the top of it
is the far clipping plane: a ray that reaches z_far reports z_far, so the
optimistic channel measures to it like it would to any other surface.

### One view, then sixteen

The same query against a buffer holding a single frame, and against the full
ring. Nothing was fused in between: the second buffer simply has more images to
search, and every one of them gets a vote.

In [ ]:
TRUNCATION = 3.0
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
for ax, count in zip(axes, (1, 4, len(poses))):
    partial = cdb.CudaDepthBuffer(intrinsics, params)
    for i in range(count):
        partial.set_posed_depth_image(depths[i], np.asarray(poses[i], np.float32),
                                      stamp_ns=i, is_keyframe=False)
    _, field, axis = query_slice(partial, truncation=TRUNCATION, height=1.0)
    draw_field(ax, field, axis, f"conservative, {count} frame(s) in the buffer",
               True, cameras=poses[:count], limit=TRUNCATION)
fig.tight_layout()

## 4. Gradients

Every query also returns the gradient of the distance, which is the direction to
push a trajectory to gain clearance. It comes from the closest point, not from
finite differences of the field, so it costs nothing extra and does not smear
across the discontinuities.

In [ ]:
step = 12
points, xx, yy, axis = slice_grid(resolution=220, height=1.0)
result = buffer.query(points, conservative=True, truncation_radius=3.0)
gradient = result.conservative_gradient.reshape(xx.shape + (3,))

fig, ax = plt.subplots(figsize=(6.4, 5.4))
draw_field(ax, conservative, axis, "conservative distance and its gradient", True,
           limit=limit)
ax.quiver(xx[::step, ::step], yy[::step, ::step],
          gradient[::step, ::step, 0], gradient[::step, ::step, 1],
          color="k", scale=28, width=0.0032)
fig.tight_layout()

## 5. Turn the knobs

Every one of these is live. Nothing is rebuilt when they change, because there
is nothing to rebuild: a redraw is sixteen inserts and one query, a few tens of
milliseconds all in.

- **drone** flies along the transect. The buffer is a ring of sixteen frames, so
  what it knows is a window that slides with the drone: space opens up ahead and
  closes again behind, and this is the whole of what a mapless method forgets.
- **truncation radius** caps the search. It is the strongest lever on query
  time, and a planner rarely cares about clearance beyond a few metres.
- **slice height** moves the plane through the scene. The trunks are vertical,
  the spheres are not, and the ground is always exactly one slice height away,
  so on a low slice nothing is ever further than that from an obstacle. The
  colour scale runs to the truncation radius either way, so a pale red means a
  metre of clearance whichever slice you are looking at.
- **robot radius** treats a ball around the camera of every frame as free, on
  the grounds that the drone was there when the image was taken. Watch the
  unknown wedge behind the flight line open up.

In [ ]:
import ipywidgets as widgets

# Its own buffer, so the robot radius below stays out of the timings further
# down: that one is buffer state, not a query argument.
live = cdb.CudaDepthBuffer(intrinsics, params)


def redraw(drone=START + WINDOW - 1, height=1.0, truncation=3.0, robot_radius=0.0,
           conservative=True):
    # The ring holds the last WINDOW frames the drone flew through. Early on
    # there are not that many yet, so the first is repeated: the buffer is full
    # of frames either way, they just do not all differ.
    window = [max(0, i) for i in range(drone - WINDOW + 1, drone + 1)]
    live.set_robot_radius(robot_radius)
    for i in window:
        live.set_posed_depth_image(flight_depths[i],
                                   np.asarray(flight[i], np.float32),
                                   stamp_ns=i, is_keyframe=False)

    points, xx, _, axis = slice_grid(resolution=200, height=height)
    result = live.query(points, conservative=conservative,
                        truncation_radius=truncation)
    field = result.distance(conservative).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    draw_field(ax, field, axis,
               f"{'conservative' if conservative else 'optimistic'}"
               f" at z = {height:.1f} m, truncated at {truncation:.1f} m",
               conservative, cameras=[flight[i] for i in window],
               limit=truncation)
    ax.plot(flight[drone][0], flight[drone][1], "^", ms=13, color="limegreen",
            markeredgecolor="k", label="drone")
    ax.legend(loc="upper right", fontsize=7)
    plt.show()


widgets.interact(
    redraw,
    drone=widgets.IntSlider(START + WINDOW - 1, min=0, max=len(flight) - 1,
                            description="drone", continuous_update=False),
    height=widgets.FloatSlider(1.0, min=0.2, max=3.5, step=0.1, description="height"),
    truncation=widgets.FloatSlider(3.0, min=0.5, max=6.0, step=0.5, description="trunc"),
    robot_radius=widgets.FloatSlider(0.0, min=0.0, max=2.0, step=0.1, description="robot r"),
    conservative=widgets.Checkbox(True, description="conservative"),
);

## 6. Keyframing

The ring is small, so which frames it holds matters. Filling it at camera rate
leaves every slot looking at nearly the same thing; filling it with frames that
disagree covers far more space.

`is_keyframe` evaluates the distance, angle and time thresholds in
`BufferParams` and **reports** the answer. It never acts on it. A caller with a
better criterion, view overlap or information gain, ignores it and passes its
own decision to `set_posed_depth_image`.

With keyframing enabled, slot 0 always holds the latest frame no matter what, so
the newest observation is never more than one insert old, while the keyframes
rotate through the remaining slots.

In [ ]:
approach = [cdb.look_at([-7.0 + 0.25 * i, -1.0 + 0.05 * i, 1.6], [0.0, 0.0, 1.0])
          for i in range(56)]

selector_params = cdb.BufferParams()
selector_params.num_frames = params.num_frames
selector_params.keyframe_dist_threshold = 1.0
selector_params.keyframe_angle_threshold = 0.15
selector_params.keyframe_time_threshold = 2.0
selector = cdb.CudaDepthBuffer(intrinsics, selector_params)

kept = []
for i, pose in enumerate(approach):
    stamp = i * 33_000_000                      # 30 Hz
    pose_array = np.asarray(pose, np.float32)
    keyframe = selector.is_keyframe(pose_array, stamp)
    if keyframe:
        kept.append(i)
    selector.set_posed_depth_image(scene.render(intrinsics, pose), pose_array,
                                   stamp_ns=stamp, is_keyframe=keyframe)

print(f"{len(kept)} of {len(approach)} poses were keyframes")

In [ ]:
def yaw_of(pose):
    qx, qy, qz, qw = pose[3:]
    return np.arctan2(2 * (qw * qz + qx * qy), 1 - 2 * (qy * qy + qz * qz))


yaw = np.unwrap([yaw_of(pose) for pose in approach])
fig, axes = plt.subplots(2, 1, figsize=(7.5, 5.2), sharex=False)

axes[0].plot([p[0] for p in approach], [p[1] for p in approach], "-", color="0.75",
             lw=2, label="pass")
axes[0].plot([approach[i][0] for i in kept], [approach[i][1] for i in kept], "o",
             ms=5, color="crimson", label="keyframe")
axes[0].plot(0.0, 0.0, "*", ms=12, color="k", label="look at")
axes[0].set_xlabel("x [m]"); axes[0].set_ylabel("y [m]")
axes[0].set_aspect("equal"); axes[0].legend(fontsize=8)
axes[0].set_title("keyframes along a second, closer pass", fontsize=10)

axes[1].plot(np.degrees(yaw), "-", color="0.75", lw=2)
axes[1].plot(kept, np.degrees(yaw)[kept], "o", ms=5, color="crimson")
axes[1].set_xlabel("frame"); axes[1].set_ylabel("camera yaw [deg]")
axes[1].set_title("and why: one per metre of travel, and more where the view swings",
                  fontsize=10)
fig.tight_layout()

The evenly spaced markers are the distance threshold firing once per metre. The
cluster is the angle threshold: flying past the point it looks at, the camera
swings through most of its yaw in a couple of frames, and each of those frames
sees something the last one did not.

Whether that is the right rule is the caller's business. `is_keyframe` only
answers the question; `set_posed_depth_image` does what it is told.

## 7. What it costs

Query time scales with the number of frames searched and with the truncation
radius, and barely at all with the number of query points once the GPU is busy.
The per stage breakdown below synchronises the device between stages, so the
parts add up to more than the whole; the split is what it is for.

In [ ]:
import time

points, _, _, _ = slice_grid(resolution=220, height=1.0)
for count in (1, 100, 10_000, points.shape[0]):
    batch = points[:count]
    buffer.query(batch, truncation_radius=3.0)          # warm the workspace
    start = time.perf_counter()
    for _ in range(20):
        buffer.query(batch, conservative=True, truncation_radius=3.0)
    elapsed = (time.perf_counter() - start) / 20
    print(f"{count:>7} queries  {1e3 * elapsed:7.3f} ms  "
          f"{1e6 * elapsed / count:7.2f} us/query")

In [ ]:
cdb.profiling.reset()
for _ in range(10):
    buffer.query(points, conservative=True, truncation_radius=3.0, verbose=True)
print(cdb.profiling.report())

## Where to go next

- `examples/cpp/concurrent_queries.cpp` runs several planning threads against one
  writer, which is what a real stack looks like.
- `benchmark/query_benchmark.cpp` reports insertion and query throughput on this
  same generated scene, so a regression can be bisected without a dataset.
- `brute_force_distances` evaluates the definition exhaustively. It is far too
  slow to use, and it is what the test suite checks the GPU against.